In [46]:

from google.colab import files
uploaded = files.upload()


Saving all files.zip to all files (4).zip


# Step 2 — Extract all ZIP files automatically

In [47]:

import os
import zipfile

main_zip = "/content/all files.zip"
extract_folder = "/content/fyp_data"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(main_zip, "r") as z:
    z.extractall(extract_folder)

# Extract the two ZIP files inside the main ZIP
for root, dirs, filenames in os.walk(extract_folder):
    for filename in filenames:
        if filename.lower().endswith(".zip"):
            zip_path = os.path.join(root, filename)
            destination = os.path.join(
                extract_folder,
                os.path.splitext(filename)[0]
            )
            os.makedirs(destination, exist_ok=True)

            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(destination)

print("All ZIP files extracted successfully.")


All ZIP files extracted successfully.


# Step 3 — Load all four datasets

In [48]:

import os
import pandas as pd

base_folder = "/content/fyp_data"
csv_files = {}

for root, dirs, filenames in os.walk(base_folder):
    for filename in filenames:
        if filename.lower().endswith(".csv"):
            key = filename.lower()
            csv_files.setdefault(key, []).append(
                os.path.join(root, filename)
            )

def load_unique_csv(filename):
    matches = csv_files.get(filename.lower(), [])

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one file named {filename}, "
            f"but found {len(matches)}. "
            f"Check the extracted files."
        )

    return pd.read_csv(matches[0])

original_engine = load_unique_csv("engine_data.csv")
clean_engine = load_unique_csv("engine_data_clean.csv")
clean_obd = load_unique_csv("carOBD_clean.csv")

# Combine the original OBD-II driving files
obd_paths = [
    path
    for name, paths in csv_files.items()
    if name.endswith(".csv")
    for path in paths
    if "/carOBD-master/obdiidata/" in path.replace("\\", "/")
]

if not obd_paths:
    raise FileNotFoundError("Original OBD-II CSV files were not found.")

original_obd = pd.concat(
    [pd.read_csv(path) for path in sorted(obd_paths)],
    ignore_index=True
)

print("Original engine:", original_engine.shape)
print("Cleaned engine:", clean_engine.shape)
print("Original OBD-II:", original_obd.shape)
print("Cleaned OBD-II:", clean_obd.shape)


Original engine: (19535, 7)
Cleaned engine: (19535, 8)
Original OBD-II: (304299, 27)
Cleaned OBD-II: (274968, 26)


# Step 4 — Compare the original and cleaned datasets

In [49]:

datasets = {
    "Original Engine": original_engine,
    "Cleaned Engine": clean_engine,
    "Original OBD-II": original_obd,
    "Cleaned OBD-II": clean_obd
}

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Rows and columns:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print("Columns:", df.columns.tolist())



--- Original Engine ---
Rows and columns: (19535, 7)
Missing values: 0
Duplicate rows: 0
Columns: ['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp', 'Engine Condition']

--- Cleaned Engine ---
Rows and columns: (19535, 8)
Missing values: 0
Duplicate rows: 0
Columns: ['engine_rpm', 'lub_oil_pressure', 'fuel_pressure', 'coolant_pressure', 'lub_oil_temp', 'coolant_temp', 'engine_condition', 'coolant_temp_extreme']

--- Original OBD-II ---
Rows and columns: (304299, 27)
Missing values: 295424
Duplicate rows: 42537
Columns: ['ENGINE_RUN_TINE ()', 'ENGINE_RPM ()', 'VEHICLE_SPEED ()', 'THROTTLE ()', 'ENGINE_LOAD ()', 'COOLANT_TEMPERATURE ()', 'LONG_TERM_FUEL_TRIM_BANK_1 ()', 'SHORT_TERM_FUEL_TRIM_BANK_1 ()', 'INTAKE_MANIFOLD_PRESSURE ()', 'FUEL_TANK ()', 'ABSOLUTE_THROTTLE_B ()', 'PEDAL_D ()', 'PEDAL_E ()', 'COMMANDED_THROTTLE_ACTUATOR ()', 'FUEL_AIR_COMMANDED_EQUIV_RATIO ()', 'ABSOLUTE_BAROMETRIC_PRESSURE ()', 'RELATIVE_THROTTLE_POSITION 

In [50]:
print("Remaining missing values:")
print(clean_obd.isnull().sum().sort_values(ascending=False).head(10))

print("\nEngine label counts:")
print(clean_engine["engine_condition"].value_counts())

print("\nCoolant extreme counts:")
print(clean_engine["coolant_temp_extreme"].value_counts())

Remaining missing values:
warm_ups_since_codes_cleared        2111
time_since_trouble_codes_cleared    2111
vehicle_speed                          0
throttle                               0
engine_load                            0
coolant_temperature                    0
long_term_fuel_trim_bank_1             0
short_term_fuel_trim_bank_1            0
engine_run_time                        0
engine_rpm                             0
dtype: int64

Engine label counts:
engine_condition
1    12317
0     7218
Name: count, dtype: int64

Coolant extreme counts:
coolant_temp_extreme
0    19533
1        2
Name: count, dtype: int64


# emaining missing values remove-save new cleaned OBD-II

In [51]:
columns_to_drop = [
    "warm_ups_since_codes_cleared",
    "time_since_trouble_codes_cleared"
]

obd_final = clean_obd.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("Final OBD-II shape:", obd_final.shape)
print("Missing values:", obd_final.isnull().sum().sum())

obd_final.to_csv(
    "/content/carOBD_final.csv",
    index=False
)

print("Saved successfully!")

Final OBD-II shape: (274968, 24)
Missing values: 0
Saved successfully!


# Sensor units check

In [52]:
print("ENGINE DATASET")
print(clean_engine.dtypes)
print(clean_engine.head())

print("\nOBD-II DATASET")
print(clean_obd.dtypes)
print(clean_obd.head())

ENGINE DATASET
engine_rpm                int64
lub_oil_pressure        float64
fuel_pressure           float64
coolant_pressure        float64
lub_oil_temp            float64
coolant_temp            float64
engine_condition          int64
coolant_temp_extreme      int64
dtype: object
   engine_rpm  lub_oil_pressure  fuel_pressure  coolant_pressure  \
0         700          2.493592      11.790927          3.178981   
1         876          2.941606      16.193866          2.464504   
2         520          2.961746       6.553147          1.064347   
3         473          3.707835      19.510172          3.727455   
4         619          5.672919      15.738871          2.052251   

   lub_oil_temp  coolant_temp  engine_condition  coolant_temp_extreme  
0     84.144163     81.632187                 1                     0  
1     77.640934     82.445724                 0                     0  
2     77.752266     79.645777                 1                     0  
3     74.129907   

In [53]:
print("Engine dataset column ranges:")
display(clean_engine.describe().T)

print("OBD-II dataset column ranges:")
display(clean_obd.describe().T)

Engine dataset column ranges:


,count,mean,std,min,25%,50%,75%,max
engine_rpm,19535.0,791.239263,267.611193,61.000000,593.000000,746.000000,934.000000,2239.000000
lub_oil_pressure,19535.0,3.303775,1.021643,0.003384,2.518815,3.162035,4.055272,7.265566
fuel_pressure,19535.0,6.655615,2.761021,0.003187,4.916886,6.201720,7.744973,21.138326
coolant_pressure,19535.0,2.335369,1.036382,0.002483,1.600466,2.166883,2.848840,7.478505
lub_oil_temp,19535.0,77.643420,3.110984,71.321974,75.725990,76.817350,78.071691,89.580796
coolant_temp,19535.0,78.427433,6.206749,61.673325,73.895421,78.346662,82.915411,195.527912
engine_condition,19535.0,0.630509,0.482679,0.000000,0.000000,1.000000,1.000000,1.000000
coolant_temp_extreme,19535.0,0.000102,0.010118,0.000000,0.000000,0.000000,0.000000,1.000000


OBD-II dataset column ranges:


,count,mean,std,min,25%,50%,75%,max
engine_run_time,274968.0,753.191626,611.291219,0.000000,293.000000,605.000000,1054.000000,3696.000000
engine_rpm,274968.0,1218.856800,488.437078,0.000000,813.250000,1122.500000,1578.750000,3768.750000
vehicle_speed,274968.0,22.924097,22.081302,0.000000,0.000000,21.000000,38.000000,127.000000
throttle,274968.0,19.255917,3.770289,13.333000,16.471000,18.823530,20.392157,78.823532
engine_load,274968.0,35.004000,14.763357,0.000000,25.490196,30.196079,38.823528,100.000000
coolant_temperature,274968.0,83.137445,11.157926,21.000000,86.000000,87.000000,88.000000,96.000000
long_term_fuel_trim_bank_1,274968.0,-0.462844,2.248675,-8.593750,-1.562500,-0.781250,0.781250,7.031200
short_term_fuel_trim_bank_1,274968.0,-0.059913,3.554728,-20.312500,-2.343750,0.000000,2.343750,19.531250
intake_manifold_pressure,274968.0,35.657000,15.103624,16.000000,26.000000,31.000000,40.000000,102.000000
fuel_tank,274968.0,47.166153,20.352610,0.000000,29.411764,43.921570,61.960785,90.980392


# Engine dataset - unusual coolanttemperature values

In [54]:

print("Highest coolant temperatures:")
display(
    clean_engine.nlargest(10, "coolant_temp")[
        ["coolant_temp", "engine_condition", "coolant_temp_extreme"]
    ]
)

print("\nLowest coolant temperatures:")
display(
    clean_engine.nsmallest(10, "coolant_temp")[
        ["coolant_temp", "engine_condition", "coolant_temp_extreme"]
    ]
)


Highest coolant temperatures:


,coolant_temp,engine_condition,coolant_temp_extreme
2423,195.527912,1,1
15123,118.371957,0,1
11754,95.855283,1,0
119,95.234554,1,0
5810,95.149178,1,0
16479,94.933593,1,0
10479,94.857819,1,0
6194,94.811546,1,0
2129,94.784392,0,0
1665,94.779375,1,0



Lowest coolant temperatures:


,coolant_temp,engine_condition,coolant_temp_extreme
10761,61.673325,1,0
4496,62.445955,0,0
4563,63.056066,1,0
12542,63.086903,1,0
5870,63.124119,1,0
5060,63.241341,1,0
3508,63.243353,1,0
7176,63.287928,1,0
6270,63.288664,1,0
13496,63.334645,1,0


In [55]:
print("Records labelled as extreme:")
display(
    clean_engine[
        clean_engine["coolant_temp_extreme"] == 1
    ][
        ["coolant_temp", "engine_condition",
         "coolant_temp_extreme"]
    ]
)

Records labelled as extreme:


,coolant_temp,engine_condition,coolant_temp_extreme
2423,195.527912,1,1
15123,118.371957,0,1


In [56]:
print("Engine condition counts:")
print(clean_engine["engine_condition"].value_counts())

print("\nCoolant extreme counts:")
print(clean_engine["coolant_temp_extreme"].value_counts())

print("\nRecords with extreme coolant temperature:")
display(
    clean_engine[
        clean_engine["coolant_temp_extreme"] == 1
    ]
)

Engine condition counts:
engine_condition
1    12317
0     7218
Name: count, dtype: int64

Coolant extreme counts:
coolant_temp_extreme
0    19533
1        2
Name: count, dtype: int64

Records with extreme coolant temperature:


,engine_rpm,lub_oil_pressure,fuel_pressure,coolant_pressure,lub_oil_temp,coolant_temp,engine_condition,coolant_temp_extreme
2423,455,2.010802,7.806127,1.619575,76.519385,195.527912,1,1
15123,760,4.908028,2.594570,1.747730,77.873482,118.371957,0,1


In [57]:

# Compare original and cleaned engine dataset

print("ORIGINAL ENGINE COLUMNS:")
print(original_engine.columns.tolist())

print("\nCLEANED ENGINE COLUMNS:")
print(clean_engine.columns.tolist())

print("\nOriginal engine shape:", original_engine.shape)
print("Cleaned engine shape:", clean_engine.shape)

print("\nOriginal dataset sample:")
display(original_engine.head())

print("\nOriginal dataset information:")
original_engine.info()


ORIGINAL ENGINE COLUMNS:
['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp', 'Engine Condition']

CLEANED ENGINE COLUMNS:
['engine_rpm', 'lub_oil_pressure', 'fuel_pressure', 'coolant_pressure', 'lub_oil_temp', 'coolant_temp', 'engine_condition', 'coolant_temp_extreme']

Original engine shape: (19535, 7)
Cleaned engine shape: (19535, 8)

Original dataset sample:


,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp,Engine Condition
0,700,2.493592,11.790927,3.178981,84.144163,81.632187,1
1,876,2.941606,16.193866,2.464504,77.640934,82.445724,0
2,520,2.961746,6.553147,1.064347,77.752266,79.645777,1
3,473,3.707835,19.510172,3.727455,74.129907,71.774629,1
4,619,5.672919,15.738871,2.052251,78.396989,87.000225,0



Original dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19535 entries, 0 to 19534
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Engine rpm        19535 non-null  int64  
 1   Lub oil pressure  19535 non-null  float64
 2   Fuel pressure     19535 non-null  float64
 3   Coolant pressure  19535 non-null  float64
 4   lub oil temp      19535 non-null  float64
 5   Coolant temp      19535 non-null  float64
 6   Engine Condition  19535 non-null  int64  
dtypes: float64(5), int64(2)
memory usage: 1.0 MB


In [58]:

# Search the notebook's current variables and label relationship

print("Original column names:")
print(original_engine.columns.tolist())

print("\nCleaned dataset columns:")
print(clean_engine.columns.tolist())

print("\nCompare coolant temperature with extreme label:")
display(
    clean_engine.groupby("coolant_temp_extreme")["coolant_temp"]
    .agg(["count", "min", "max", "mean"])
)


Original column names:
['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp', 'Engine Condition']

Cleaned dataset columns:
['engine_rpm', 'lub_oil_pressure', 'fuel_pressure', 'coolant_pressure', 'lub_oil_temp', 'coolant_temp', 'engine_condition', 'coolant_temp_extreme']

Compare coolant temperature with extreme label:


,count,min,max,mean
coolant_temp_extreme,,,,
0,19533,61.673325,95.855283,78.419393
1,2,118.371957,195.527912,156.949934


In [59]:

# Check whether the extreme label matches a simple temperature threshold

display(
    clean_engine[
        ["coolant_temp", "coolant_temp_extreme"]
    ].sort_values("coolant_temp").iloc[
        [0, 1, -3, -2, -1]
    ]
)

print("Unique extreme labels:")
print(clean_engine["coolant_temp_extreme"].unique())

print("\nRows where extreme label is 1:")
display(
    clean_engine.loc[
        clean_engine["coolant_temp_extreme"] == 1,
        ["coolant_temp", "engine_condition",
         "coolant_temp_extreme"]
    ]
)


,coolant_temp,coolant_temp_extreme
10761,61.673325,0
4496,62.445955,0
11754,95.855283,0
15123,118.371957,1
2423,195.527912,1


Unique extreme labels:
[0 1]

Rows where extreme label is 1:


,coolant_temp,engine_condition,coolant_temp_extreme
2423,195.527912,1,1
15123,118.371957,0,1


In [60]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "engine_data_clean.csv":
            print("Found file:", os.path.join(root, file))

Found file: /content/fyp_data/all files/engine_data_clean.csv


In [61]:
import pandas as pd

engine = pd.read_csv(
    "/content/fyp_data/all files/engine_data_clean.csv"
)

print("Engine condition counts:")
print(engine["engine_condition"].value_counts(dropna=False))

print("\nCoolant temperature by engine condition:")
display(
    engine.groupby("engine_condition")["coolant_temp"]
    .agg(["count", "min", "max", "mean"])
)

Engine condition counts:
engine_condition
1    12317
0     7218
Name: count, dtype: int64

Coolant temperature by engine condition:


,count,min,max,mean
engine_condition,,,,
0,7218,62.445955,118.371957,78.803030
1,12317,61.673325,195.527912,78.207326


In [62]:
print("Engine dataset column ranges:")
display(engine.describe().T)

print("\nOBD-II dataset column ranges:")
obd = pd.read_csv(
    "/content/fyp_data/all files/carOBD_clean.csv"
)
display(obd.describe().T)

Engine dataset column ranges:


,count,mean,std,min,25%,50%,75%,max
engine_rpm,19535.0,791.239263,267.611193,61.000000,593.000000,746.000000,934.000000,2239.000000
lub_oil_pressure,19535.0,3.303775,1.021643,0.003384,2.518815,3.162035,4.055272,7.265566
fuel_pressure,19535.0,6.655615,2.761021,0.003187,4.916886,6.201720,7.744973,21.138326
coolant_pressure,19535.0,2.335369,1.036382,0.002483,1.600466,2.166883,2.848840,7.478505
lub_oil_temp,19535.0,77.643420,3.110984,71.321974,75.725990,76.817350,78.071691,89.580796
coolant_temp,19535.0,78.427433,6.206749,61.673325,73.895421,78.346662,82.915411,195.527912
engine_condition,19535.0,0.630509,0.482679,0.000000,0.000000,1.000000,1.000000,1.000000
coolant_temp_extreme,19535.0,0.000102,0.010118,0.000000,0.000000,0.000000,0.000000,1.000000



OBD-II dataset column ranges:


,count,mean,std,min,25%,50%,75%,max
engine_run_time,274968.0,753.191626,611.291219,0.000000,293.000000,605.000000,1054.000000,3696.000000
engine_rpm,274968.0,1218.856800,488.437078,0.000000,813.250000,1122.500000,1578.750000,3768.750000
vehicle_speed,274968.0,22.924097,22.081302,0.000000,0.000000,21.000000,38.000000,127.000000
throttle,274968.0,19.255917,3.770289,13.333000,16.471000,18.823530,20.392157,78.823532
engine_load,274968.0,35.004000,14.763357,0.000000,25.490196,30.196079,38.823528,100.000000
coolant_temperature,274968.0,83.137445,11.157926,21.000000,86.000000,87.000000,88.000000,96.000000
long_term_fuel_trim_bank_1,274968.0,-0.462844,2.248675,-8.593750,-1.562500,-0.781250,0.781250,7.031200
short_term_fuel_trim_bank_1,274968.0,-0.059913,3.554728,-20.312500,-2.343750,0.000000,2.343750,19.531250
intake_manifold_pressure,274968.0,35.657000,15.103624,16.000000,26.000000,31.000000,40.000000,102.000000
fuel_tank,274968.0,47.166153,20.352610,0.000000,29.411764,43.921570,61.960785,90.980392


# Finnaly....

### Dataset Verification and Label Limitations

**Engine Health Dataset**
- Source: [Kaggle Engine Health Dataset](https://www.kaggle.com/datasets/parvmodi/automotive-vehicles-engine-health-dataset)
- Target column: `engine_condition`
- The dataset contains engine RPM, lubrication pressure, fuel pressure, coolant pressure, lubrication temperature and coolant temperature.
- The meaning of the original `engine_condition` labels should be confirmed from the dataset documentation before training.
- The additional `coolant_temp_extreme` column is absent from the original dataset. Its generation rule is unknown; therefore, it will be excluded from model training.

**OBD-II Dataset**
- Source: [willkn/odb2 GitHub repository](https://github.com/willkn/odb2)
- Vehicle: 2014 Toyota Etios; sampling frequency: 1 Hz.
- The dataset contains engine coolant temperature, RPM, vehicle speed, engine load and other OBD-II parameters.
- The repository identifies the parameters, but not every unit should be inferred from descriptive statistics alone.

**Data limitation:** Neither dataset provides verified, separate ground-truth labels for both selected fault categories: engine overheating and low engine oil pressure. Any rule-generated labels will be clearly identified as simulated labels, not real observed failures.

----------------------------------------------------------------------

In [63]:
import os
import pandas as pd

os.makedirs("/content/initial_datasets", exist_ok=True)

# Engine dataset: exclude the unverified extra label
engine_initial = clean_engine.drop(
    columns=["coolant_temp_extreme"],
    errors="ignore"
).copy()

# OBD-II dataset: use the cleaned final version
obd_initial = obd_final.copy()

engine_initial.to_csv(
    "/content/initial_datasets/engine_initial.csv",
    index=False
)

obd_initial.to_csv(
    "/content/initial_datasets/obd_initial.csv",
    index=False
)

print("Engine initial dataset:", engine_initial.shape)
print("Engine missing values:", engine_initial.isnull().sum().sum())

print("OBD-II initial dataset:", obd_initial.shape)
print("OBD-II missing values:", obd_initial.isnull().sum().sum())

print("Initial datasets saved successfully.")

Engine initial dataset: (19535, 7)
Engine missing values: 0
OBD-II initial dataset: (274968, 24)
OBD-II missing values: 0
Initial datasets saved successfully.


In [64]:
from google.colab import files

files.download(
    "/content/initial_datasets/engine_initial.csv"
)

files.download(
    "/content/initial_datasets/obd_initial.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Dataset Inspection and Verification

### 3.1 Engine Health Dataset

**Source:** [Automotive Vehicles Engine Health Dataset – Kaggle](https://www.kaggle.com/datasets/parvmodi/automotive-vehicles-engine-health-dataset)

- Number of records: 19,535
- Number of columns in the original dataset: 7
- Missing values: 0
- Sensor features: Engine RPM, lubrication oil pressure, fuel pressure, coolant pressure, lubrication oil temperature, and coolant temperature.
- Target column: `Engine Condition`.
- Label counts: `1` = 12,317 records; `0` = 7,218 records.
- A secondary technical analysis identifies label `1` as Good and label `0` as Bad. However, this mapping has not been independently confirmed from the original dataset documentation.
- The additional column `coolant_temp_extreme` was excluded because its generation rule could not be verified.
- Pressure measurement units have not been conclusively confirmed from the original dataset documentation.

### 3.2 OBD-II Dataset

**Source:** [willkn/obd2 – GitHub](https://github.com/willkn/odb2)

- Number of records in the prepared dataset: 274,968
- Number of columns in the prepared dataset: 24
- Missing values in the prepared dataset: 0
- Available signals include engine RPM, vehicle speed, engine load, coolant temperature, and throttle-related measurements.
- The source vehicle is a 2014 Toyota Etios, not a Toyota Aqua/Prius C.
- This dataset will be used as supporting vehicle telemetry data, not as direct validation of Toyota Aqua fault conditions.

### Step 3 Final Verification Record

The columns, record counts, missing values, and label distributions of both datasets have been inspected.

- **Engine Health Dataset:** Contains 19,535 records and six sensor features with the `Engine Condition` target. Secondary sources report conflicting interpretations of the target labels. Therefore, the meanings of labels 0 and 1 remain unconfirmed for this project.
- **Pressure units:** The original dataset documentation available for this project does not conclusively confirm the units of the lubrication oil pressure, fuel pressure, or coolant pressure columns. These units are recorded as unverified.
- **OBD-II Dataset:** The source README identifies the data as measurements collected from a 2014 Toyota Etios at 1 Hz. Standard OBD-II parameter definitions can be used to document units where the corresponding parameter mapping is confirmed. The dataset is not Toyota Aqua/Prius C fault-labelled data.
- **Prepared data:** The cleaned datasets have been saved. The additional `coolant_temp_extreme` column was excluded because its generation rule could not be verified.

**Conclusion:** Dataset inspection is complete. Unresolved label meanings and pressure units are explicitly documented as limitations and will not be assumed during model training or threshold design.

## 4. Toyota Aqua/Prius C Technical References

The following Toyota Prius C service-manual references will guide the definition of fault conditions in the software prototype.

1. **Coolant temperature and sensor diagnostics:** [Toyota coolant temperature diagnostic procedure](https://lemon.dogeware.me/Toyota/2016/Prius%20C%20One/Repair%20and%20Diagnosis%20%28Single%20Page%29/Engine%20Performance/System/SFI%20System%20-%20Diagnostic%20Codes%20%283%20Of%204%29/SFI%20System/DTC%20P0115%3A%20Engine%20Coolant%20Temperature%20Circuit%3B%20DTC%20P0117%3A%20Engine%20Coolant%20Temperature%20Circuit%20Low%20Input%3B%20DTC%20P0118%3A%20Engine%20Coolant%20Temperature%20Circuit%20High%20Input%20%5B11%2F2015%20-%2006%2F2017%5D/Procedure/)

2. **Engine oil pressure inspection:** [Toyota lubrication-system inspection procedure](https://lemon.dogeware.me/Toyota/2016/Prius%20C%20One/Repair%20and%20Diagnosis%20%28Single%20Page%29/Engine%20Mechanical/Lubrication%20System/Engine%20Lubrication%20System%20%28Inspection%29/Lubrication%20System/On-Vehicle%20Inspection%20%5B11%2F2015%20-%20%5D/Procedure/)

### 4.1 Reference Values

- **Warmed-up engine coolant temperature:** 75–100°C.
- **Minimum engine oil pressure at idle:** 59 kPa.
- **Engine oil pressure at 2,500 RPM:** 140–280 kPa.

These values apply under the operating conditions specified in the service procedures.

### 4.2 Application to This Project

- Coolant temperature specifications will guide the design of simulated overheating scenarios.
- Oil-pressure specifications will guide the design of simulated low-oil-pressure scenarios.
- Sensor-circuit faults will be considered separately and will not automatically be classified as engine overheating.
- A coolant reading above 135°C may indicate a sensor-circuit fault. The sensor condition must be investigated separately from an actual overheating condition.
- Numerical thresholds will not be applied directly to public dataset measurements until their units, operating conditions, and limitations are considered.

### 4.3 Limitations

The Toyota service-manual values are reference specifications, not automatically validated thresholds for the public datasets. The Engine Health Dataset's pressure units and the exact operating conditions of its records remain unconfirmed.

The OBD-II dataset was collected from a Toyota Etios rather than a Toyota Aqua/Prius C. Therefore, it provides supporting telemetry but does not establish vehicle-specific fault labels.

These references support the design of the software simulation. They do not establish that the public datasets contain verified Toyota Aqua/Prius C fault labels.

## 5. Selected Fault Categories

Based on the approved project proposal, the prototype focuses on two engine-related fault categories.

### Class 1: Engine Overheating

- **Relevant signal:** Engine coolant temperature.
- **Supporting signals:** Engine RPM, engine load, and vehicle operating conditions, where available.
- **Data strategy:** Use verified technical specifications to design gradual, simulated overheating sequences. Sensor-circuit faults will be treated separately from actual overheating scenarios.

### Class 2: Low Engine Oil Pressure

- **Relevant signal:** Lubrication oil pressure.
- **Supporting signals:** Engine RPM and vehicle operating conditions, where available.
- **Data strategy:** Verify the pressure units in the available dataset and the relevant Toyota service specifications before designing simulated low-oil-pressure sequences.

### Label Availability and Limitations

The available datasets do not provide confirmed, separate ground-truth labels for both selected fault categories. The existing dataset labels will be kept distinct from the synthetic labels generated for simulated fault scenarios.

Synthetic fault labels will be created using documented rules and will be clearly identified as simulated labels. They will not be presented as confirmed real-world vehicle failures. Any assumptions, threshold values, and label-generation rules will be documented before model training.

The project scope remains limited to engine overheating and low engine oil pressure, in line with the approved proposal. A third fault category will not be added unless approved by the project supervisor.

### Step 5 Scope Confirmation

The project supervisor has confirmed that the prototype will focus on the two fault categories defined in the approved project proposal:

1. Engine overheating
2. Low engine oil pressure

A third fault category is not required for the current project scope. Data preparation, synthetic fault generation, model development, and evaluation will focus on these two categories.

**Conclusion:** The fault-category selection is finalized in accordance with the approved proposal and supervisor confirmation.

## 6. Data Cleaning and Initial Dataset Preparation

### Engine Health Dataset
- Loaded the cleaned engine dataset.
- Excluded the unverified `coolant_temp_extreme` column from the initial modelling dataset.
- Saved the prepared dataset as `engine_initial.csv`.
- Final shape: 19,535 rows × 7 columns.
- Missing values: 0.

### OBD-II Dataset
- Removed duplicate records during the cleaning process.
- Removed `warm_ups_since_codes_cleared` and `time_since_trouble_codes_cleared` because these columns contained missing values and were not required for the selected initial features.
- Saved the prepared dataset as `obd_initial.csv`.
- Final shape: 274,968 rows × 24 columns.
- Missing values: 0.

### Output and Limitations

Both initial datasets have been uploaded to the GitHub repository. The datasets are ready for further feature selection and baseline modelling. Fault-specific labels and sensor units must be verified or explicitly defined before training the fault prediction models.

In [67]:
import pandas as pd

# Use the already loaded dataset in memory
df = engine_initial

print("Columns:")
print(df.columns.tolist())

print("\nEngine condition label counts:")
print(df["engine_condition"].value_counts(dropna=False))

print("\nEngine condition vs sensor statistics:")
print(
    df.groupby("engine_condition")[
        ["lub_oil_pressure", "coolant_temp", "engine_rpm"]
    ].agg(["min", "median", "max"])
)

print("\nPressure values:")
print(df["lub_oil_pressure"].describe())

Columns:
['engine_rpm', 'lub_oil_pressure', 'fuel_pressure', 'coolant_pressure', 'lub_oil_temp', 'coolant_temp', 'engine_condition']

Engine condition label counts:
engine_condition
1    12317
0     7218
Name: count, dtype: int64

Engine condition vs sensor statistics:
                 lub_oil_pressure                     coolant_temp             \
                              min    median       max          min     median   
engine_condition                                                                
0                        0.007891  3.067195  7.051322    62.445955  78.768801   
1                        0.003384  3.214466  7.265566    61.673325  78.100543   

                             engine_rpm               
                         max        min median   max  
engine_condition                                      
0                 118.371957        351  843.0  2239  
1                 195.527912         61  690.0  2172  

Pressure values:
count    19535.000000
mean     